# Manuka-0527 GPU Training

Run this notebook in a GPU Colab runtime after the CPU prepare notebook. It copies prepared artifacts from Drive to local Colab disk, trains the scratch GPT model, and saves the best model back to Drive.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Run once per fresh Colab runtime if sentencepiece is missing.
try:
    import sentencepiece  # noqa: F401
except ImportError:
    %pip -q install sentencepiece


In [3]:
from pathlib import Path
import gzip
import json
import shutil
from collections import Counter

DATA_ROOT = Path('/content/drive/MyDrive/text_dataset/manuka/manuka-0527')
PREP_DIR = DATA_ROOT / 'prepared'
MODEL_DIR = DATA_ROOT / 'manuka-model'
LOCAL_PREP_DIR = Path('/content/manuka-0527-prepared')

if not PREP_DIR.exists():
    raise FileNotFoundError(f'{PREP_DIR} does not exist. Run the CPU prepare notebook first.')
if LOCAL_PREP_DIR.exists():
    shutil.rmtree(LOCAL_PREP_DIR)
shutil.copytree(PREP_DIR, LOCAL_PREP_DIR)
WORK_PREP_DIR = LOCAL_PREP_DIR
MODEL_DIR.mkdir(parents=True, exist_ok=True)

with open(WORK_PREP_DIR / 'data_manifest.json', encoding='utf-8') as f:
    manifest = json.load(f)
MAX_LEN = int(manifest['context_length'])

with gzip.open(WORK_PREP_DIR / manifest.get('tokenized_file', 'tokenized_examples.jsonl.gz'), 'rt', encoding='utf-8') as f:
    tokenized_examples = [json.loads(line) for line in f]

print('Loaded prepared data from local disk:', WORK_PREP_DIR)
print('Original Drive prep dir:', PREP_DIR)
print('Tokenized rows:', len(tokenized_examples))
print('MAX_LEN:', MAX_LEN)


Loaded prepared data from local disk: /content/manuka-0527-prepared
Original Drive prep dir: /content/drive/MyDrive/text_dataset/manuka/manuka-0527/prepared
Tokenized rows: 9298863
MAX_LEN: 512


In [4]:
import re
import sentencepiece as spm

sp = spm.SentencePieceProcessor(model_file=str(WORK_PREP_DIR / manifest.get('tokenizer_model', 'spm16k.model')))
PAD = sp.pad_id(); BOS = sp.bos_id(); EOS = sp.eos_id(); UNK = sp.unk_id()
SEP = sp.piece_to_id('<sep>')
VOCAB_SIZE = sp.get_piece_size()
SPECIALS = {'PAD': PAD, 'BOS': BOS, 'SEP': SEP, 'EOS': EOS, 'UNK': UNK}
SPACE_RGX = re.compile(r'\s+')


def clean_text(s: str) -> str:
    s = (s or '').replace('​', ' ').replace('﻿', ' ').strip()
    return SPACE_RGX.sub(' ', s)


def encode_text(s: str):
    return sp.encode(clean_text(s), out_type=int)


def decode_text(ids):
    drop = {PAD, BOS, SEP, EOS, UNK}
    return sp.decode([int(i) for i in ids if int(i) not in drop])

print('Tokenizer vocab size:', VOCAB_SIZE, SPECIALS)


Tokenizer vocab size: 16000 {'PAD': 0, 'BOS': 1, 'SEP': 2, 'EOS': 3, 'UNK': 4}


In [5]:
kind_token_counts = Counter()
tokens_per_epoch = 0
for ex in tokenized_examples:
    tokens_per_epoch += len(ex['input_ids'])
    kind_token_counts[ex.get('kind', 'unknown')] += len(ex['input_ids'])

print(f'Examples per epoch: {len(tokenized_examples)}')
print(f'Tokens per epoch before dynamic padding: {tokens_per_epoch}')
print('Token mix:', dict(kind_token_counts))


Examples per epoch: 9298863
Tokens per epoch before dynamic padding: 149944111
Token mix: {'next_2': 74856938, 'next_1': 75087173}


# Transformer


In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint


def rotate_half(x):
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)


def apply_rope(q, k, *, rope_freqs, start_pos=0):
    if q.shape[-1] != k.shape[-1]:
        raise ValueError("q and k must have the same head dimension")

    original_dim = q.shape[-1]
    head_dim = original_dim + original_dim % 2
    if original_dim % 2 != 0:
        q = F.pad(q, (0, 1))
        k = F.pad(k, (0, 1))

    seq_len = q.shape[-2]
    freqs = rope_freqs[start_pos:start_pos + seq_len, :head_dim].to(device=q.device, dtype=torch.float32)
    cos = freqs.cos().to(dtype=q.dtype).view(1, 1, seq_len, head_dim)
    sin = freqs.sin().to(dtype=q.dtype).view(1, 1, seq_len, head_dim)

    q_out = (q * cos) + (rotate_half(q) * sin)
    k_out = (k * cos) + (rotate_half(k) * sin)
    if original_dim % 2 != 0:
        q_out = q_out[..., :original_dim]
        k_out = k_out[..., :original_dim]
    return q_out.contiguous(), k_out.contiguous()


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_float = x.float()
        normed = x_float * torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return normed.to(dtype=x.dtype) * self.weight.to(dtype=x.dtype)


class CausalSelfAttention(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        dropout,
        max_len,
        n_kv_heads=None,
        qk_norm=True,
        rope_theta=10000.0,
        rope_scale=1.0,
        rope_scaling_type="linear",
    ):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.n_kv_heads = n_heads if n_kv_heads is None else n_kv_heads
        if n_heads % self.n_kv_heads != 0:
            raise ValueError("n_heads must be divisible by n_kv_heads")
        if rope_scale <= 0:
            raise ValueError("rope_scale must be positive")
        if rope_scaling_type not in {"linear", "ntk"}:
            raise ValueError("rope_scaling_type must be 'linear' or 'ntk'")

        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // self.n_kv_heads
        self.rope_theta = rope_theta
        self.rope_scale = rope_scale
        self.rope_scaling_type = rope_scaling_type

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.head_dim) if qk_norm else nn.Identity()
        self.k_norm = RMSNorm(self.head_dim) if qk_norm else nn.Identity()
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer("rope_freqs", self._build_rope_freqs(max_len), persistent=False)

    def _build_rope_freqs(self, max_len):
        head_dim = self.head_dim + self.head_dim % 2
        fractions = torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
        inv_freq = 1.0 / (self.rope_theta ** fractions)
        if self.rope_scaling_type == "ntk":
            inv_freq = inv_freq / (self.rope_scale ** fractions)
            positions = torch.arange(max_len, dtype=torch.float32)
        else:
            positions = torch.arange(max_len, dtype=torch.float32) / self.rope_scale
        freqs = torch.einsum("i,j->ij", positions, inv_freq)
        return torch.repeat_interleave(freqs, 2, dim=1)

    def _repeat_kv(self, x):
        if self.kv_repeat == 1:
            return x
        return x.repeat_interleave(self.kv_repeat, dim=1)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)
        past_len = past_kv[0].shape[-2] if past_kv is not None else 0
        start_pos = past_len
        q, k = apply_rope(q, k, rope_freqs=self.rope_freqs, start_pos=start_pos)

        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)
        present = (k, v) if use_cache else None

        k_full = self._repeat_kv(k)
        v_full = self._repeat_kv(v)
        key_len = k_full.shape[-2]

        causal_needed = T > 1 or past_kv is None
        attn_bias = None
        if attn_mask is not None:
            if attn_mask.dim() != 2:
                raise ValueError("attn_mask must have shape (batch, key_length)")
            if attn_mask.shape[-1] != key_len:
                attn_mask = attn_mask[:, -key_len:]
            pad_mask = (attn_mask == 0).unsqueeze(1).unsqueeze(2)
            attn_bias = torch.zeros((B, 1, T, key_len), device=x.device, dtype=q.dtype)
            attn_bias = attn_bias.masked_fill(pad_mask, torch.finfo(q.dtype).min)
            if causal_needed:
                q_pos = torch.arange(start_pos, start_pos + T, device=x.device).view(T, 1)
                k_pos = torch.arange(key_len, device=x.device).view(1, key_len)
                causal_mask = k_pos > q_pos
                attn_bias = attn_bias.masked_fill(causal_mask.view(1, 1, T, key_len), torch.finfo(q.dtype).min)
                causal_needed = False

        y = F.scaled_dot_product_attention(
            q,
            k_full,
            v_full,
            attn_mask=attn_bias,
            dropout_p=self.attn_drop.p if self.training else 0.0,
            is_causal=causal_needed,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.out_proj(y))
        return y, present


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))


class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        mlp_ratio=8/3,
        dropout=0.1,
        max_len=MAX_LEN,
        n_kv_heads=None,
        qk_norm=True,
        residual_scale=1.0,
        rope_theta=10000.0,
        rope_scale=1.0,
        rope_scaling_type="linear",
    ):
        super().__init__()
        hidden_dim = int(mlp_ratio * d_model)
        hidden_dim = 256 * math.ceil(hidden_dim / 256)
        self.ln1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(
            d_model,
            n_heads,
            dropout,
            max_len,
            n_kv_heads=n_kv_heads,
            qk_norm=qk_norm,
            rope_theta=rope_theta,
            rope_scale=rope_scale,
            rope_scaling_type=rope_scaling_type,
        )
        self.ln2 = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, hidden_dim, dropout)
        self.attn_res_scale = nn.Parameter(torch.tensor(float(residual_scale)))
        self.mlp_res_scale = nn.Parameter(torch.tensor(float(residual_scale)))

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        attn_out, present = self.attn(self.ln1(x), attn_mask=attn_mask, past_kv=past_kv, use_cache=use_cache)
        x = x + self.attn_res_scale * attn_out
        x = x + self.mlp_res_scale * self.mlp(self.ln2(x))
        return x, present


class GPTScratch(nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        d_model=512,
        n_layers=10,
        n_heads=8,
        n_kv_heads=None,
        max_len=MAX_LEN,
        dropout=0.1,
        qk_norm=True,
        residual_scale=1.0,
        rope_theta=10000.0,
        rope_scale=1.0,
        rope_scaling_type="linear",
        gradient_checkpointing=False,
    ):
        super().__init__()
        self.max_len = max_len
        self.n_layers = n_layers
        self.gradient_checkpointing = gradient_checkpointing
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.emb_norm = RMSNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                n_heads,
                mlp_ratio=8/3,
                dropout=dropout,
                max_len=max_len,
                n_kv_heads=n_kv_heads,
                qk_norm=qk_norm,
                residual_scale=residual_scale,
                rope_theta=rope_theta,
                rope_scale=rope_scale,
                rope_scaling_type=rope_scaling_type,
            )
            for _ in range(n_layers)
        ])
        self.ln_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init)
        self._scale_residual_projections()
        self.head.weight = self.tok_emb.weight

    def _init(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _scale_residual_projections(self):
        scale = 0.02 / math.sqrt(2 * self.n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scale)
            nn.init.normal_(block.mlp.w3.weight, mean=0.0, std=scale)

    def forward(self, x, attention_mask=None, past_kv=None, use_cache=False):
        B, T = x.shape
        if past_kv is None and T > self.max_len:
            x = x[:, -self.max_len:]
            T = x.shape[1]
            if attention_mask is not None:
                attention_mask = attention_mask[:, -self.max_len:]

        h = self.drop(self.emb_norm(self.tok_emb(x)))
        presents = [] if use_cache else None
        if past_kv is None:
            past_kv = [None] * len(self.blocks)

        for block, layer_past in zip(self.blocks, past_kv):
            if self.gradient_checkpointing and self.training and not use_cache:
                h, present = checkpoint.checkpoint(block, h, attention_mask, layer_past, False, use_reentrant=False)
            else:
                h, present = block(h, attn_mask=attention_mask, past_kv=layer_past, use_cache=use_cache)
            if use_cache:
                presents.append(present)

        logits = self.head(self.ln_f(h))
        if use_cache:
            return logits, presents
        return logits

# Training


In [8]:
import torch, torch.nn.functional as F, time, json, os, math, random, shutil
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Built model structure; no external model libraries.
D_MODEL = 512
N_LAYERS = 10
N_HEADS = 8
N_KV_HEADS = 8       # keep full MHA quality by default; set 4 for GQA speed/memory tradeoff
QK_NORM = True       # normalizes q/k per head before RoPE for stabler attention logits
RESIDUAL_SCALE = 1.0 # learnable per-block residual branch scale starts neutral
ROPE_THETA = 1000000.0
ROPE_SCALE = 1.0
ROPE_SCALING = "linear"
MODEL_DROPOUT = 0.08
GRADIENT_CHECKPOINTING = False

model = GPTScratch(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS, n_kv_heads=N_KV_HEADS,
    max_len=MAX_LEN, dropout=MODEL_DROPOUT,
    qk_norm=QK_NORM, residual_scale=RESIDUAL_SCALE,
    rope_theta=ROPE_THETA, rope_scale=ROPE_SCALE, rope_scaling_type=ROPE_SCALING,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
).to(device)

BASE_LR = 3e-4
MIN_LR = 3e-5
WEIGHT_DECAY = 0.10
LABEL_SMOOTHING = 0.02
BATCH_SIZE = 768
ACCUM = 4
EPOCHS = 10
PATIENCE = 5
VAL_FRACTION = 0.02
USE_COMPILE = False
save_dir = str(MODEL_DIR)

if USE_COMPILE and hasattr(torch, "compile"):
    try:
        model = torch.compile(model)
        print("torch.compile enabled")
    except Exception as exc:
        print("torch.compile skipped:", repr(exc))

optimizer_kwargs = dict(lr=BASE_LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95), eps=1e-8)
if device == "cuda":
    optimizer_kwargs["fused"] = True
try:
    optimizer = torch.optim.AdamW(model.parameters(), **optimizer_kwargs)
except TypeError:
    optimizer_kwargs.pop("fused", None)
    optimizer = torch.optim.AdamW(model.parameters(), **optimizer_kwargs)

amp_dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))


def lm_loss(logits, targets):
    vocab = logits.size(-1)
    pred = logits[:, :-1].contiguous().view(-1, vocab)
    gold = targets[:, 1:].contiguous().view(-1)
    return F.cross_entropy(pred, gold, ignore_index=-100, label_smoothing=LABEL_SMOOTHING)


class ConversationDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        return row["input_ids"], row["labels"]


def collate_batch(batch, pad_to_multiple_of=8):
    max_len = max(len(x[0]) for x in batch)
    if pad_to_multiple_of:
        max_len = int(math.ceil(max_len / pad_to_multiple_of) * pad_to_multiple_of)
    input_ids, labels, attention_mask = [], [], []
    for ids, lbl in batch:
        pad_len = max_len - len(ids)
        input_ids.append(ids + [PAD] * pad_len)
        labels.append(lbl + [-100] * pad_len)
        attention_mask.append([1] * len(ids) + [0] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }

rows = tokenized_examples[:]
rng = random.Random(SEED)
rng.shuffle(rows)
val_size = max(1, int(len(rows) * VAL_FRACTION))
val_rows = rows[:val_size]
train_rows = rows[val_size:]

train_dataset = ConversationDataset(train_rows)
val_dataset = ConversationDataset(val_rows)
loader_kwargs = dict(num_workers=2 if device == "cuda" else 0, pin_memory=torch.cuda.is_available(), collate_fn=collate_batch)
if loader_kwargs["num_workers"] > 0:
    loader_kwargs["persistent_workers"] = True
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, **loader_kwargs)
val_dl = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

updates_per_epoch = math.ceil(len(train_dl) / ACCUM)
TOTAL_STEPS = EPOCHS * updates_per_epoch
WARMUP_STEPS = max(25, int(0.06 * TOTAL_STEPS))


def lr_lambda(step):
    min_ratio = MIN_LR / BASE_LR
    if step < WARMUP_STEPS:
        return max(min_ratio, float(step + 1) / float(WARMUP_STEPS))
    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return min_ratio + (1.0 - min_ratio) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def evaluate(max_batches=None):
    model.eval(); total_loss, total_tokens = 0.0, 0
    for bi, b in enumerate(val_dl, 1):
        x = b["input_ids"].to(device); y = b["labels"].to(device); attn = b["attention_mask"].to(device)
        with torch.amp.autocast(device_type="cuda", dtype=amp_dtype, enabled=(device == "cuda")):
            loss = lm_loss(model(x, attention_mask=attn), y)
        target_tokens = (y[:, 1:] != -100).sum().item()
        total_loss += loss.item() * max(1, target_tokens)
        total_tokens += max(1, target_tokens)
        if max_batches and bi >= max_batches:
            break
    return total_loss / max(1, total_tokens)


def raw_model_state_dict():
    raw = getattr(model, "_orig_mod", model)
    return raw.state_dict()

print(f"Train examples: {len(train_dataset)} | Val examples: {len(val_dataset)} | Updates/epoch: {updates_per_epoch}")

best = float("inf")
bad_epochs = 0
os.makedirs(save_dir, exist_ok=True)
for src_name in ['spm16k.model', 'spm16k.vocab', 'data_manifest.json']:
    src_path = WORK_PREP_DIR / src_name
    if src_path.exists():
        shutil.copy2(src_path, Path(save_dir) / src_name)

for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); optimizer.zero_grad(set_to_none=True)
    running_loss, running_count = 0.0, 0
    for i, b in enumerate(train_dl, 1):
        x = b["input_ids"].to(device, non_blocking=True)
        y = b["labels"].to(device, non_blocking=True)
        attn = b["attention_mask"].to(device, non_blocking=True)
        group_start = ((i - 1) // ACCUM) * ACCUM + 1
        group_end = min(group_start + ACCUM - 1, len(train_dl))
        accum_scale = group_end - group_start + 1
        with torch.amp.autocast(device_type="cuda", dtype=amp_dtype, enabled=(device == "cuda")):
            loss = lm_loss(model(x, attention_mask=attn), y) / accum_scale
        scaler.scale(loss).backward()
        running_loss += loss.item() * accum_scale
        running_count += 1
        if i % ACCUM == 0 or i == len(train_dl):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            step = (ep - 1) * updates_per_epoch + math.ceil(i / ACCUM)
            if step % 100 == 0:
                print(f"step {step} | loss {running_loss / max(1, running_count):.4f} | lr {scheduler.get_last_lr()[0]:.2e}")
                running_loss, running_count = 0.0, 0

    val = evaluate()
    print(f"[epoch {ep}] val_loss={val:.4f} | {(time.time() - t0) / 60:.1f} min")
    if val < best:
        best = val
        bad_epochs = 0
        torch.save(raw_model_state_dict(), f"{save_dir}/model.pt")
        with open(f"{save_dir}/config.json", "w", encoding="utf-8") as f:
            json.dump({
                "vocab_size": VOCAB_SIZE, "d_model": D_MODEL, "n_layers": N_LAYERS,
                "n_heads": N_HEADS, "n_kv_heads": N_KV_HEADS,
                "max_len": MAX_LEN, "dropout": MODEL_DROPOUT,
                "qk_norm": QK_NORM, "residual_scale": RESIDUAL_SCALE,
                "rope_theta": ROPE_THETA, "rope_scale": ROPE_SCALE, "rope_scaling_type": ROPE_SCALING,
                "gradient_checkpointing": False,
                "special_tokens": SPECIALS,
                "tokenizer_model": "spm16k.model",
                "generation": {"max_new_tokens": 96, "temperature": 0.72, "top_p": 0.88, "top_k": 60, "repetition_penalty": 1.12, "no_repeat_ngram_size": 4},
                "optimizer": {"base_lr": BASE_LR, "min_lr": MIN_LR, "weight_decay": WEIGHT_DECAY, "label_smoothing": LABEL_SMOOTHING},
                "training": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "accum": ACCUM, "warmup_steps": WARMUP_STEPS, "val_fraction": VAL_FRACTION, "best_val_loss": best},
                "paths": {"data_root": str(DATA_ROOT), "prepared_dir": str(PREP_DIR), "model_dir": str(MODEL_DIR)},
            }, f, ensure_ascii=False, indent=2)
        print("SAVED:", save_dir)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stop after {PATIENCE} epochs without validation improvement")
            break


/tmp/ipykernel_3091/2628502505.py:66: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda" and amp_dtype == torch.float16))


Train examples: 9112886 | Val examples: 185977 | Updates/epoch: 2967
step 100 | loss 8.4267 | lr 3.00e-05
step 200 | loss 7.0502 | lr 3.39e-05
step 300 | loss 6.6115 | lr 5.07e-05
step 400 | loss 6.3135 | lr 6.76e-05
step 500 | loss 6.0842 | lr 8.44e-05
step 600 | loss 5.9106 | lr 1.01e-04
step 700 | loss 5.7805 | lr 1.18e-04
step 800 | loss 5.6682 | lr 1.35e-04
step 900 | loss 5.5685 | lr 1.52e-04
step 1000 | loss 5.4820 | lr 1.69e-04
step 1100 | loss 5.4012 | lr 1.86e-04
step 1200 | loss 5.3274 | lr 2.02e-04
step 1300 | loss 5.2645 | lr 2.19e-04
step 1400 | loss 5.2005 | lr 2.36e-04
step 1500 | loss 5.1387 | lr 2.53e-04
step 1600 | loss 5.0922 | lr 2.70e-04
step 1700 | loss 5.0446 | lr 2.87e-04
step 1800 | loss 5.0019 | lr 3.00e-04
step 1900 | loss 4.9622 | lr 3.00e-04
step 2000 | loss 4.9228 | lr 3.00e-04
step 2100 | loss 4.8908 | lr 3.00e-04
step 2200 | loss 4.8646 | lr 3.00e-04
step 2300 | loss 4.8326 | lr 3.00e-04
step 2400 | loss 4.8098 | lr 3.00e-04
step 2500 | loss 4.7859 | lr

KeyboardInterrupt: 

# Zip Saved Model


In [9]:
from pathlib import Path
import shutil

src_dir = MODEL_DIR
assert src_dir.is_dir(), f'{src_dir} folder does not exist. Train or copy a model there first.'

zip_base = DATA_ROOT / 'manuka-model'
zip_file = Path(f'{zip_base}.zip')
if zip_file.exists():
    zip_file.unlink()

zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format='zip',
    root_dir=src_dir.parent,
    base_dir=src_dir.name,
)
print('Saved zip:', zip_path)


Saved zip: /content/drive/MyDrive/text_dataset/manuka/manuka-0527/manuka-model.zip
